# Handwritten Digit Classification with PyTorch

This notebook presents handwritten digit classification on the MNIST dataset using several neural network architectures implemented in PyTorch.

The project compares:
- a single-layer Perceptron
- a Deep fully-connected network
- a basic CNN
- a tuned BetterCNN model

The main goals are to compare model performance, analyze overfitting, apply regularization, and tune a convolutional network to achieve more than **99% test accuracy**.

## Project goals

- load and preprocess the MNIST dataset
- split the data into training, validation, and test subsets
- train and compare multiple neural network models
- analyze overfitting using training and validation loss
- apply regularization techniques
- tune a CNN model to exceed 99% test accuracy
- visualize model errors with a confusion matrix

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

plt.rcParams.update({'font.size': 14})

model_args = {
    'seed': 123,
    'batch_size': 128,
    'lr': 0.05,
    'momentum': 0.5,
    'epochs': 50,
    'log_interval': 100
}

torch.manual_seed(model_args['seed'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

ModuleNotFoundError: No module named 'torch'

## Dataset

The MNIST dataset contains grayscale handwritten digit images of size **28 × 28** pixels.

It consists of:
- **60,000** training images
- **10,000** test images
- **10 classes** representing digits from `0` to `9`

In this project, the original training set is split into:
- **50,000** training samples
- **10,000** validation samples

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_train = datasets.MNIST('../data', train=True, download=True, transform=transform)
train_subset, validation_subset = torch.utils.data.random_split(mnist_train, [50000, 10000])
test_subset = datasets.MNIST('../data', train=False, download=True, transform=transform)

loader_kwargs = {
    'batch_size': model_args['batch_size'],
    'num_workers': 2,
    'pin_memory': True
}

train_loader = torch.utils.data.DataLoader(train_subset, shuffle=True, **loader_kwargs)
validation_loader = torch.utils.data.DataLoader(validation_subset, shuffle=False, **loader_kwargs)
test_loader = torch.utils.data.DataLoader(test_subset, shuffle=False, **loader_kwargs)

print(f"Train size: {len(train_subset)}")
print(f"Validation size: {len(validation_subset)}")
print(f"Test size: {len(test_subset)}")
print(f"Number of train batches: {len(train_loader)}")

## Sample images from MNIST

In [ ]:
example_number = 123

fig, axs = plt.subplots(5, 5, figsize=(7, 7), tight_layout=True)
for i in range(5):
    for j in range(5):
        image, label = train_subset[example_number + i * 5 + j]
        axs[i, j].imshow(image.squeeze(0), cmap='gray')
        axs[i, j].set_title(label)
        axs[i, j].axis('off')

plt.show()

## Model definitions

We begin with a simple Perceptron baseline, then test a deeper fully-connected network, and finally compare them with convolutional models.

### Perceptron

The Perceptron is the simplest baseline model.  
Each image is flattened into a 784-dimensional vector and passed directly to a linear classification layer.

In [ ]:
class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(28 * 28, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

### Deep fully-connected network

This model adds a hidden layer and a ReLU activation, which increases model capacity compared with the single-layer Perceptron.

In [ ]:
class Deep(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 20)
        self.fc2 = nn.Linear(20, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

### Basic CNN

The CNN model is better suited for image classification because it preserves spatial structure and learns local visual patterns.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 2, kernel_size=5)
        self.conv2 = nn.Conv2d(2, 4, kernel_size=5)
        self.fc1 = nn.Linear(64, 20)
        self.fc2 = nn.Linear(20, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(-1, 64)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

### BetterCNN

This tuned CNN is deeper and wider than the original version.  
It uses:
- more convolution filters
- Batch Normalization
- dropout
- Adam optimizer

This model is used to exceed **99% test accuracy**.

In [ ]:
class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, 10)

        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # 28 -> 14
        x = self.pool(F.relu(self.bn2(self.conv2(x))))   # 14 -> 7
        x = self.pool(F.relu(self.bn3(self.conv3(x))))   # 7 -> 3

        x = self.dropout1(x)
        x = x.view(-1, 128 * 3 * 3)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)

        return F.log_softmax(x, dim=1)

## Training and evaluation utilities

In [ ]:
def train(model, device, train_loader, optimizer, epoch_number, log_interval=100):
    model.train()
    train_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target, reduction='mean')
        loss.backward()
        optimizer.step()

        if batch_idx % log_interval == 0:
            print(
                f'Train Epoch: {epoch_number} '
                f'[{batch_idx * len(data)}/{len(train_loader.dataset)} '
                f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}'
            )

        train_loss += loss.item()

    train_loss /= len(train_loader)
    print(f'\nTrain set: Average loss: {train_loss:.4f}')
    return train_loss


def evaluate(model, device, data_loader, message='Evaluation set'):
    model.eval()
    total_loss = 0.0
    correct = 0

    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)

            total_loss += F.nll_loss(output, target, reduction='mean').item()
            prediction = output.argmax(dim=1)
            correct += prediction.eq(target).sum().item()

    total_loss /= len(data_loader)
    accuracy = 100.0 * correct / len(data_loader.dataset)

    print(f'{message}: Average loss: {total_loss:.4f}, Accuracy: {correct}/{len(data_loader.dataset)} ({accuracy:.0f}%)\n')
    return total_loss, accuracy


def plot_losses(train_losses, validation_losses, title):
    plt.figure(figsize=(8, 5))
    plt.grid(True)
    plt.xlabel("Epoch")
    plt.ylabel("Average loss")
    plt.plot(range(1, len(train_losses) + 1), train_losses, 'o-', label='training')
    plt.plot(range(1, len(validation_losses) + 1), validation_losses, 'o-', label='validation')
    plt.legend()
    plt.title(title)
    plt.show()

In [ ]:
def run_experiment(model, optimizer, epochs, title):
    train_losses = []
    validation_losses = []

    for epoch_number in range(1, epochs + 1):
        train_losses.append(
            train(model, device, train_loader, optimizer, epoch_number, model_args['log_interval'])
        )
        val_loss, _ = evaluate(model, device, validation_loader, 'Validation set')
        validation_losses.append(val_loss)

    test_loss, test_acc = evaluate(model, device, test_loader, 'Test set')
    plot_losses(train_losses, validation_losses, title)

    return {
        'train_losses': train_losses,
        'validation_losses': validation_losses,
        'test_loss': test_loss,
        'test_acc': test_acc
    }

## Perceptron experiment

The Perceptron serves as a simple linear baseline.

In [ ]:
torch.manual_seed(model_args['seed'])
perceptron_model = Perceptron().to(device)

perceptron_optimizer = optim.SGD(
    perceptron_model.parameters(),
    lr=model_args['lr'],
    momentum=model_args['momentum']
)

perceptron_results = run_experiment(
    perceptron_model,
    perceptron_optimizer,
    model_args['epochs'],
    'Perceptron model'
)

The Perceptron achieved about **92%** test accuracy and served as a useful baseline, but its capacity was limited and it showed mild overfitting.

## Deep fully-connected model

Adding a hidden layer improves the model capacity and classification performance.

In [ ]:
torch.manual_seed(model_args['seed'])
deep_model = Deep().to(device)

deep_optimizer = optim.SGD(
    deep_model.parameters(),
    lr=model_args['lr'],
    momentum=model_args['momentum']
)

deep_results = run_experiment(
    deep_model,
    deep_optimizer,
    model_args['epochs'],
    'Deep model'
)

The Deep model improved the result to about **96%** accuracy, but the widening gap between training and validation performance indicated overfitting.

## Deep model with L2 regularization

To reduce overfitting, L2 regularization was introduced through the `weight_decay` parameter in the SGD optimizer.

In [ ]:
torch.manual_seed(model_args['seed'])
deep_l2_model = Deep().to(device)

deep_l2_optimizer = optim.SGD(
    deep_l2_model.parameters(),
    lr=model_args['lr'],
    momentum=model_args['momentum'],
    weight_decay=1e-3
)

deep_l2_results = run_experiment(
    deep_l2_model,
    deep_l2_optimizer,
    model_args['epochs'],
    'Deep model with L2 regularization'
)

L2 regularization made the validation loss more stable and slightly reduced overfitting, while preserving the test accuracy at about **96%**.

## Original CNN model

The CNN model is better suited to image data because it preserves spatial structure and learns local patterns.

In [ ]:
torch.manual_seed(model_args['seed'])
cnn_model = CNN().to(device)

cnn_optimizer = optim.SGD(
    cnn_model.parameters(),
    lr=model_args['lr'],
    momentum=model_args['momentum']
)

cnn_results = run_experiment(
    cnn_model,
    cnn_optimizer,
    model_args['epochs'],
    'CNN model'
)

The CNN generalized better than the fully-connected models.  
In this model, the validation loss could become lower than the training loss because dropout was active during training and disabled during evaluation.

## Tuned BetterCNN model

A deeper and wider CNN with Batch Normalization, dropout, and Adam optimizer was used to exceed **99% test accuracy**.

In [ ]:
torch.manual_seed(model_args['seed'])
better_cnn_model = BetterCNN().to(device)

better_cnn_optimizer = optim.Adam(
    better_cnn_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

better_cnn_results = run_experiment(
    better_cnn_model,
    better_cnn_optimizer,
    15,
    'Better CNN model'
)

The tuned BetterCNN achieved **99.27%** test accuracy on MNIST, exceeding the required **99% threshold**.

## Confusion matrix

The confusion matrix shows how often the final tuned model confuses one digit with another.
Rows represent the true labels, while columns represent the predicted labels.

In [ ]:
better_cnn_model.eval()

all_targets = []
all_predictions = []

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = better_cnn_model(data)
        prediction = output.argmax(dim=1)

        all_targets.extend(target.cpu().numpy())
        all_predictions.extend(prediction.cpu().numpy())

cm = confusion_matrix(all_targets, all_predictions)

print("Confusion matrix:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title("Confusion Matrix - Test Set")
plt.show()

Most values lie on the main diagonal, which confirms very strong performance.

The most frequent errors occur between visually similar digits, such as:
- **9 and 5**
- **9 and 4**
- **4 and 9**
- **7 and 2**

This suggests that the remaining mistakes are mainly caused by similarities in handwritten digit shapes.

## Conclusions

- The **Perceptron** achieved about **92%** test accuracy and served as a simple baseline.
- The **Deep** model improved the result to about **96%**, but showed clear signs of overfitting.
- **L2 regularization** slightly reduced overfitting in the Deep model.
- **CNN-based models** performed better because they were more suitable for image data.
- The tuned **BetterCNN** achieved **99.27%** test accuracy, exceeding the required **99% threshold**.
- The confusion matrix confirmed that the remaining mistakes occurred mainly between visually similar handwritten digits.## Conclusions

- The **Perceptron** achieved about **92%** test accuracy and served as a simple baseline.
- The **Deep** model improved the result to about **96%**, but showed clear signs of overfitting.
- **L2 regularization** slightly reduced overfitting in the Deep model.
- **CNN-based models** performed better because they were more suitable for image data.
- The tuned **BetterCNN** achieved **99.27%** test accuracy, exceeding the required **99% threshold**.
- The confusion matrix confirmed that the remaining mistakes occurred mainly between visually similar handwritten digits.